In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

# Original toy corpus
corpus = """
We are using PyTorch to build a deep learning language model.
PyTorch makes it easy to construct neural networks and train them on data.
This n-gram language model learns to predict the next word based on context.
""".lower().split()

CONTEXT_SIZE = 2
EMBEDDING_DIM = 10
HIDDEN_DIM = 32

# Keep these specific to the toy model
toy_vocab = set(corpus)
toy_vocab_size = len(toy_vocab)
toy_word_to_ix = {word: i for i, word in enumerate(toy_vocab)}
toy_ix_to_word = {i: word for i, word in enumerate(toy_vocab)}

trigrams = [
    ([corpus[i], corpus[i + 1]], corpus[i + 2])
    for i in range(len(corpus) - CONTEXT_SIZE)
]

print(f"Toy Model Vocabulary Size: {toy_vocab_size}")

Toy Model Vocabulary Size: 32


In [11]:
class NGramLanguageModeler(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, vocab_size)
        self.relu = nn.ReLU()

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = self.relu(self.linear1(embeds))
        out = self.linear2(out)
        return out

# Instantiate the toy model
toy_model = NGramLanguageModeler(toy_vocab_size, EMBEDDING_DIM, CONTEXT_SIZE, HIDDEN_DIM)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(toy_model.parameters(), lr=0.1)

# Train toy model
epochs = 50
for epoch in range(epochs):
    total_loss = 0
    for context, target in trigrams:
        context_idxs = torch.tensor([toy_word_to_ix[w] for w in context], dtype=torch.long)
        toy_model.zero_grad()
        log_probs = toy_model(context_idxs)
        loss = loss_function(log_probs, torch.tensor([toy_word_to_ix[target]], dtype=torch.long))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

In [12]:
def predict_toy_word(context_words):
    context_words = [w.lower() for w in context_words]
    if len(context_words) != CONTEXT_SIZE:
        print(f"Error: Context must be exactly {CONTEXT_SIZE} words.")
        return
    for w in context_words:
        if w not in toy_word_to_ix:
            print(f"Error: '{w}' is not in the toy vocabulary.")
            return

    with torch.no_grad():
        inputs = torch.tensor([toy_word_to_ix[w] for w in context_words], dtype=torch.long)
        output = toy_model(inputs)
        predicted_idx = torch.argmax(output, dim=1).item()
        print(f"Context: {context_words} -> Predicted Next Word: '{toy_ix_to_word[predicted_idx]}'")

print("--- Toy Model Predictions ---")
predict_toy_word(["we", "are"])
predict_toy_word(["pytorch", "makes"])

--- Toy Model Predictions ---
Context: ['we', 'are'] -> Predicted Next Word: 'using'
Context: ['pytorch', 'makes'] -> Predicted Next Word: 'it'


In [5]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11756 sha256=b6bb00f4a6cd05443ad945c95cfd16266f938c9c8aaeab082d1e51b127f14fa4
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


In [13]:
import wikipedia
import re

try:
    wiki_page = wikipedia.page("Artificial intelligence")
    raw_text = wiki_page.content
except Exception as e:
    print(f"Error fetching Wikipedia page: {e}")
    raw_text = "Artificial intelligence is intelligence demonstrated by machines, as opposed to intelligence of humans."

cleaned_text = re.sub(r'[^a-zA-Z\s]', '', raw_text).lower()
large_corpus = cleaned_text.split()

# Unique vocabulary mappings for the Wiki model
wiki_vocab = set(large_corpus)
wiki_vocab_size = len(wiki_vocab)
wiki_word_to_ix = {word: i for i, word in enumerate(wiki_vocab)}
wiki_ix_to_word = {i: word for i, word in enumerate(wiki_vocab)}

large_trigrams = [
    ([large_corpus[i], large_corpus[i + 1]], large_corpus[i + 2])
    for i in range(len(large_corpus) - CONTEXT_SIZE)
]

Error fetching Wikipedia page: Expecting value: line 1 column 1 (char 0)


In [14]:
large_model = NGramLanguageModeler(wiki_vocab_size, EMBEDDING_DIM, CONTEXT_SIZE, HIDDEN_DIM)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(large_model.parameters(), lr=0.01)

train_data = large_trigrams[:3000]

print("Training on Wikipedia subset...")
for epoch in range(15):
    total_loss = 0
    for context, target in train_data:
        context_idxs = torch.tensor([wiki_word_to_ix[w] for w in context], dtype=torch.long)
        large_model.zero_grad()
        log_probs = large_model(context_idxs)
        loss = loss_function(log_probs, torch.tensor([wiki_word_to_ix[target]], dtype=torch.long))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

Training on Wikipedia subset...


In [15]:
def predict_wiki_word(context_words):
    context_words = [w.lower() for w in context_words]
    for w in context_words:
        if w not in wiki_word_to_ix:
            print(f"Error: '{w}' not in Wikipedia corpus vocabulary.")
            return

    with torch.no_grad():
        inputs = torch.tensor([wiki_word_to_ix[w] for w in context_words], dtype=torch.long)
        output = large_model(inputs)
        predicted_idx = torch.argmax(output, dim=1).item()
        print(f"Context: {context_words} -> Predicted Next Word: '{wiki_ix_to_word[predicted_idx]}'")

print("--- Predictions on Wikipedia Model ---")
predict_wiki_word(["artificial", "intelligence"])
predict_wiki_word(["by", "machines"])

--- Predictions on Wikipedia Model ---
Context: ['artificial', 'intelligence'] -> Predicted Next Word: 'is'
Context: ['by', 'machines'] -> Predicted Next Word: 'as'


In [16]:
print("=== Additional Toy Model Predictions ===")
predict_toy_word(["learning", "language"])
predict_toy_word(["easy", "to"])
predict_toy_word(["neural", "networks"])

print("\n=== Additional Wikipedia Model Predictions ===")
predict_wiki_word(["intelligence", "demonstrated"])
predict_wiki_word(["opposed", "to"])
predict_wiki_word(["intelligence", "of"])

=== Additional Toy Model Predictions ===
Context: ['learning', 'language'] -> Predicted Next Word: 'model.'
Context: ['easy', 'to'] -> Predicted Next Word: 'construct'
Context: ['neural', 'networks'] -> Predicted Next Word: 'and'

=== Additional Wikipedia Model Predictions ===
Context: ['intelligence', 'demonstrated'] -> Predicted Next Word: 'by'
Context: ['opposed', 'to'] -> Predicted Next Word: 'intelligence'
Context: ['intelligence', 'of'] -> Predicted Next Word: 'humans'


In [17]:
def generate_paragraph(model, word_to_ix, ix_to_word, start_context, max_words=20):
    # Standardize start context
    context = [w.lower() for w in start_context]
    generated_words = list(context)

    # Validate starting context
    for w in context:
        if w not in word_to_ix:
            print(f"Error: '{w}' is not in the vocabulary.")
            return

    # Iteratively predict the next word
    for _ in range(max_words):
        with torch.no_grad():
            inputs = torch.tensor([word_to_ix[w] for w in context[-CONTEXT_SIZE:]], dtype=torch.long)
            output = model(inputs)
            predicted_idx = torch.argmax(output, dim=1).item()
            next_word = ix_to_word[predicted_idx]
            generated_words.append(next_word)
            context.append(next_word)

    # Print the resulting generated sequence as a paragraph
    paragraph = " ".join(generated_words)
    print(f"Generated Paragraph:\n\" {paragraph} ... \"\n")

print("--- Generating Paragraph from Toy Model ---")
generate_paragraph(toy_model, toy_word_to_ix, toy_ix_to_word, ["we", "are"], max_words=15)

print("--- Generating Paragraph from Wikipedia Model ---")
generate_paragraph(large_model, wiki_word_to_ix, wiki_ix_to_word, ["artificial", "intelligence"], max_words=10)

--- Generating Paragraph from Toy Model ---
Generated Paragraph:
" we are using pytorch to build a deep learning language model. pytorch makes it easy to construct ... "

--- Generating Paragraph from Wikipedia Model ---
Generated Paragraph:
" artificial intelligence is intelligence demonstrated by machines as opposed to intelligence of ... "

